In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = Path("DataCoSupplyChainDataset.csv")
CLEAN_FILE = Path("DataCo_Cleaned_Interim.csv")

print("Raw dataset:", raw.shape)
print("Cleaned dataset:", clean.shape)

Raw dataset: (159928, 53)
Cleaned dataset: (180519, 53)


In [37]:
checks = []

def add_check(area, check, result, status, notes=""):
    checks.append({
        "Validation Area": area,
        "Check": check,
        "Result": result,
        "Status": status,
        "Notes": notes
    })

add_check(
    "Structure", "Raw dataset dimensions",
    f"{raw.shape[0]} rows × {raw.shape[1]} columns", "INFO"
)

add_check(
    "Structure", "Cleaned dataset dimensions",
    f"{clean.shape[0]} rows × {clean.shape[1]} columns", "INFO"
)

add_check(
    "Structure", "Row count preserved",
    f"{len(raw)} → {len(clean)}",
    "PASS" if len(raw) == len(clean) else "REVIEW"
)

same_columns = raw.columns.tolist() == clean.columns.tolist()

add_check(
    "Structure", "Column names and order preserved",
    same_columns, "PASS" if same_columns else "REVIEW"
)

add_check(
    "Duplicates", "Raw exact duplicate rows",
    int(raw.duplicated().sum()), "INFO"
)

add_check(
    "Duplicates", "Cleaned exact duplicate rows",
    int(clean.duplicated().sum()),
    "PASS" if clean.duplicated().sum() == 0 else "REVIEW"
)

In [38]:
missing_report = pd.DataFrame({
    "Column": raw.columns,
    "Raw Missing": raw.isna().sum().reindex(raw.columns).values,
    "Cleaned Missing": clean.isna().sum().reindex(raw.columns).values
})

missing_report["Difference"] = (
    missing_report["Cleaned Missing"] - missing_report["Raw Missing"]
)

numeric_results = []

for col in raw.select_dtypes(include="number").columns:
    if col not in clean.columns:
        failures = "Column missing"
        status = "REVIEW"
    else:
        values = clean[col]
        converted = pd.to_numeric(values, errors="coerce")
        failures = int((values.notna() & converted.isna()).sum())
        status = "PASS" if failures == 0 else "REVIEW"

    numeric_results.append({
        "Column": col,
        "Conversion Failures": failures,
        "Status": status
    })

numeric_report = pd.DataFrame(numeric_results)

for _, row in numeric_report.iterrows():
    add_check(
        "Numeric",
        f"Numeric conversion: {row['Column']}",
        row["Conversion Failures"],
        row["Status"]
    )

In [39]:
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

date_results = []

for col in date_columns:
    if col not in clean.columns:
        failures = "Column missing"
        status = "REVIEW"
    else:
        parsed = pd.to_datetime(clean[col], errors="coerce")
        failures = int((clean[col].notna() & parsed.isna()).sum())
        status = "PASS" if failures == 0 else "REVIEW"

    date_results.append({
        "Column": col,
        "Parse Failures": failures,
        "Status": status
    })

    add_check("Dates", f"Date parsing: {col}", failures, status)

order_col, ship_col = date_columns

if order_col in clean.columns and ship_col in clean.columns:
    order_dates = pd.to_datetime(clean[order_col], errors="coerce")
    ship_dates = pd.to_datetime(clean[ship_col], errors="coerce")

    valid = order_dates.notna() & ship_dates.notna()
    before = int((ship_dates[valid] < order_dates[valid]).sum())

    add_check(
        "Dates", "Shipping before order date",
        before, "PASS" if before == 0 else "REVIEW"
    )

date_report = pd.DataFrame(date_results)

In [40]:
transformed_columns = {
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Customer Id",
    "Order Customer Id",
    "Product Card Id",
    "Order Item Cardprod Id",
    "Category Id",
    "Department Id",
    "Product Category Id",
    "Customer Zipcode",
    "Order Zipcode"
}

preservation_results = []

if len(raw) != len(clean):
    preservation_results.append({
        "Column": "ALL UNTRANSFORMED COLUMNS",
        "Rows Compared": 0,
        "Mismatches": "Not checked: row counts differ",
        "Status": "REVIEW",
        "Notes": "Resolve row count differences before row-wise comparison."
    })
else:
    for col in raw.columns:
        if col not in clean.columns or col in transformed_columns:
            continue

        # Normalize surrounding whitespace only for comparison.
        a = raw[col].astype("string").str.strip()
        b = clean[col].astype("string").str.strip()

        equal = (a.eq(b) | (a.isna() & b.isna())).fillna(False)
        mismatches = int((~equal).sum())

        preservation_results.append({
            "Column": col,
            "Rows Compared": len(raw),
            "Mismatches": mismatches,
            "Status": "PASS" if mismatches == 0 else "REVIEW",
            "Notes": "Compared after trimming surrounding whitespace."
        })

preservation_report = pd.DataFrame(preservation_results)

if len(raw) != len(clean):
    add_check(
        "Value Preservation",
        "Untransformed column values",
        "Not checked: row counts differ",
        "REVIEW"
    )
else:
    mismatch_columns = (
        int((preservation_report["Status"] == "REVIEW").sum())
        if not preservation_report.empty else 0
    )

    add_check(
        "Value Preservation",
        "Columns with mismatched values",
        mismatch_columns,
        "PASS" if mismatch_columns == 0 else "REVIEW"
    )

In [41]:
category_columns = [
    "Category Id", "Category Name",
    "Department Id", "Department Name",
    "Product Card Id", "Product Name"
]

category_report = pd.DataFrame([
    {
        "Column": col,
        "Unique Values": clean[col].nunique(dropna=True),
        "Missing Values": clean[col].isna().sum()
    }
    for col in category_columns if col in clean.columns
])

mapping_pairs = [
    ("Category Id", "Category Name"),
    ("Department Id", "Department Name"),
    ("Product Card Id", "Product Name")
]

mapping_results = []

for id_col, name_col in mapping_pairs:
    if id_col in clean.columns and name_col in clean.columns:
        counts = clean.groupby(id_col, dropna=False)[name_col].nunique()
        conflicts = int((counts > 1).sum())

        mapping_results.append({
            "ID Column": id_col,
            "Name Column": name_col,
            "IDs With Multiple Names": conflicts,
            "Status": "PASS" if conflicts == 0 else "REVIEW"
        })

mapping_report = pd.DataFrame(mapping_results)

postal_columns = [
    c for c in ["Customer Zipcode", "Order Zipcode"]
    if c in clean.columns
]

postal_report = pd.DataFrame([
    {
        "Column": col,
        "Data Type": str(clean[col].dtype),
        "Missing Values": int(clean[col].isna().sum()),
        "Unique Values": int(clean[col].nunique(dropna=True))
    }
    for col in postal_columns
])

for _, row in mapping_report.iterrows():
    add_check(
        "Mapping",
        f"{row['ID Column']} to {row['Name Column']}",
        row["IDs With Multiple Names"],
        row["Status"],
        "Review conflicts; do not automatically alter source mappings."
    )

In [42]:
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

checks_report = pd.DataFrame(checks)

# Separate cleaning log: issues, evidence, decisions and status.
cleaning_log = pd.DataFrame([
    {
        "Issue": "Row count difference",
        "Evidence": f"Raw: {len(raw)} rows; cleaned: {len(clean)} rows",
        "Decision": "Investigate any difference before approval.",
        "Status": "REVIEW" if len(raw) != len(clean) else "CHECKED"
    },
    {
        "Issue": "ID-to-label mapping conflicts",
        "Evidence": (
            "See the Mapping Review sheet in the validation report."
        ),
        "Decision": "Do not auto-correct without source evidence.",
        "Status": (
            "OPEN - review required"
            if not mapping_report.empty
            and (mapping_report["Status"] == "REVIEW").any()
            else "CHECKED"
        )
    },
    {
        "Issue": "Postal-code formatting",
        "Evidence": "Postal codes should be preserved as identifiers.",
        "Decision": "Keep as text when exporting; do not invent missing zeros.",
        "Status": "DOCUMENTED"
    },
    {
        "Issue": "Value preservation",
        "Evidence": "See Value Preservation in the validation report.",
        "Decision": "Review mismatches before approving the cleaned data.",
        "Status": (
            "REVIEW" if len(raw) != len(clean)
            or (
                not preservation_report.empty
                and (preservation_report["Status"] == "REVIEW").any()
            )
            else "CHECKED"
        )
    },
    {
        "Issue": "Duplicate rows",
        "Evidence": (
            f"Raw duplicates: {raw.duplicated().sum()}; "
            f"cleaned duplicates: {clean.duplicated().sum()}"
        ),
        "Decision": "Report duplicates; do not remove automatically.",
        "Status": "DOCUMENTED"
    }
])

def format_workbook(writer):
    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(
                horizontal="center", vertical="center", wrap_text=True
            )
            cell.fill = PatternFill(
                "solid", fgColor="D9EAF7"
            )

        for column_cells in ws.columns:
            letter = get_column_letter(column_cells[0].column)
            width = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in column_cells
            )
            ws.column_dimensions[letter].width = min(max(width + 2, 12), 55)

        ws.row_dimensions[1].height = 30

with pd.ExcelWriter(
    "Cleaning_Validation_Report_Final.xlsx",
    engine="openpyxl"
) as writer:
    checks_report.to_excel(writer, sheet_name="Checks", index=False)
    preservation_report.to_excel(
        writer, sheet_name="Value Preservation", index=False
    )
    missing_report.to_excel(writer, sheet_name="Missing Values", index=False)
    numeric_report.to_excel(writer, sheet_name="Numeric Checks", index=False)
    date_report.to_excel(writer, sheet_name="Date Checks", index=False)
    category_report.to_excel(writer, sheet_name="Category Review", index=False)
    mapping_report.to_excel(writer, sheet_name="Mapping Review", index=False)
    postal_report.to_excel(writer, sheet_name="Postal Codes", index=False)
    format_workbook(writer)

with pd.ExcelWriter(
    "Updated_Cleaning_Log_Final.xlsx",
    engine="openpyxl"
) as writer:
    cleaning_log.to_excel(writer, sheet_name="Cleaning Log", index=False)
    format_workbook(writer)

print("Created: Cleaning_Validation_Report_Final.xlsx")
print("Created: Updated_Cleaning_Log_Final.xlsx")
print("Raw dataset unchanged.")
display(checks_report)
display(cleaning_log)

Created: Cleaning_Validation_Report_Final.xlsx
Created: Updated_Cleaning_Log_Final.xlsx
Raw dataset unchanged.


,Validation Area,Check,Result,Status,Notes
0,Structure,Raw dataset dimensions,159928 rows × 53 columns,INFO,
1,Structure,Cleaned dataset dimensions,180519 rows × 53 columns,INFO,
2,Structure,Row count preserved,159928 → 180519,REVIEW,
3,Structure,Column names and order preserved,True,PASS,
4,Duplicates,Raw exact duplicate rows,0,INFO,
5,Duplicates,Cleaned exact duplicate rows,0,PASS,
6,Numeric,Numeric conversion: Days for shipping (real),0,PASS,
7,Numeric,Numeric conversion: Days for shipment (scheduled),0,PASS,
8,Numeric,Numeric conversion: Benefit per order,0,PASS,
9,Numeric,Numeric conversion: Sales per customer,0,PASS,


,Issue,Evidence,Decision,Status
0,Row count difference,Raw: 159928 rows; cleaned: 180519 rows,Investigate any difference before approval.,REVIEW
1,ID-to-label mapping conflicts,See the Mapping Review sheet in the validation...,Do not auto-correct without source evidence.,CHECKED
2,Postal-code formatting,Postal codes should be preserved as identifiers.,Keep as text when exporting; do not invent mis...,DOCUMENTED
3,Value preservation,See Value Preservation in the validation report.,Review mismatches before approving the cleaned...,REVIEW
4,Duplicate rows,Raw duplicates: 0; cleaned duplicates: 0,Report duplicates; do not remove automatically.,DOCUMENTED
